<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_6/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_6_4_%D0%90%D0%B3%D0%B5%D0%BD%D1%82_%D0%BD%D0%B0_LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 6.4. Агент на LangGraph

## Введение: от цепочек к графам состояний

В Лекции 6.3 мы сделали огромный шаг вперёд: перешли от ручного управления каждым компонентом RAG-системы к элегантным цепочкам LangChain. Мы научились собирать пайплайны с помощью оператора `|`, добавлять память и парсить структурированные ответы. Код стал декларативным, читаемым и компактным — в 3–5 раз короче, чем в Лекции 6.2.

Но есть одна проблема: **цепочки линейны**. Они всегда выполняют шаги в одном и том же порядке. Если нам нужно принять решение, выполнить действие, оценить результат и, возможно, повторить его с другими параметрами — цепочка бессильна. Представьте агента, который должен:

1. Найти в документах информацию о продукте.
2. Если информации недостаточно — поискать в интернете.
3. Если ответ требует вычислений — вызвать калькулятор.
4. Если результат выглядит сомнительно — спросить у пользователя.

Такое поведение требует **ветвлений, циклов и управления состоянием** — всего того, что не умеют линейные цепочки.

Здесь на сцену выходит **LangGraph** — библиотека от создателей LangChain, предназначенная для построения **графов состояний**. В таком графе:

- **Узлы (nodes)** — это функции, которые выполняют какое-то действие (вызов LLM, поиск, вычисление).
- **Рёбра (edges)** — это правила, которые определяют, куда перейти дальше, основываясь на текущем состоянии.

Состояние — это словарь (или TypedDict), который путешествует по графу и накапливает информацию: сообщения диалога, результаты поиска, промежуточные вычисления, флаги и т.д. Каждый узел может читать и изменять состояние, а рёбра решают, какой узел будет следующим.

В этой лекции мы построим **настоящего агента** — систему, которая:
- умеет искать в документах (RAG) через ретривер,
- умеет вычислять (калькулятор),
- умеет искать в интернете (через DuckDuckGo или другой API),
- умеет определять текущее время,
- может спросить пользователя, если не уверена в действии.

Мы научим агента **циклически** принимать решения: он будет анализировать вопрос, выбирать инструмент, выполнять его, анализировать результат и решать, нужно ли повторить или завершить работу. Всё это — локально, бесплатно и с полным контролем.

---

## Тема 1. Зачем нужен LangGraph (вместо простой цепочки)

Прежде чем писать код, давайте разберёмся, почему одного LCEL недостаточно для создания полноценного агента, и как LangGraph решает эти проблемы.

### 1.1. Ограничения линейных цепочек

В Лекции 6.3 мы строили цепочки вида:

```python
chain = prompt | llm | parser
```

Или более сложные:

```python
rag_chain = (
    RunnablePassthrough.assign(context=retriever)
    | prompt
    | llm
    | parser
)
```

Это отлично работает, когда последовательность действий известна заранее и не меняется. Но что, если мы хотим реализовать такой сценарий:

**Задача:** пользователь спрашивает: «Какая средняя выручка компании за последние три квартала?»

**Что должен сделать агент:**
1. Понять, что нужны данные из документов.
2. Найти в документах выручку за Q1, Q2, Q3.
3. Если какой-то квартал отсутствует — поискать в интернете.
4. Вычислить среднее арифметическое (калькулятор).
5. Вернуть ответ.

В линейной цепочке мы не можем:
- **Ветвиться** — выбирать между поиском в документах и интернете в зависимости от результата.
- **Циклиться** — повторять поиск с разными запросами, пока не найдём все данные.
- **Останавливаться досрочно** — если данных достаточно, не тратить ресурсы на лишние действия.

**Пример, когда цепочка ломается:**

```python
# Мы не можем написать такой код в LCEL:
if answer_is_incomplete:
    search_internet()
else:
    calculate_average()
```

LCEL не поддерживает условные операторы и циклы. Он предназначен для **однозаходовых** пайплайнов, где путь данных жёстко задан.

### 1.2. Понятие графа состояний

LangGraph предлагает другой подход: мы строим **граф**, где:

- **Узлы (nodes)** — это отдельные функции, каждая из которых выполняет одно действие.
- **Состояние (state)** — это словарь или TypedDict, который хранит все данные, накопленные за время работы агента.
- **Рёбра (edges)** — это функции, которые принимают текущее состояние и возвращают имя следующего узла.

Вот как выглядит базовый граф для нашего будущего агента:

```
         ┌─────────────────┐
         │   Начало        │
         │ (получить вопрос)│
         └────────┬────────┘
                  ▼
         ┌─────────────────┐
         │   Маршрутизатор  │
         │ (выбрать действие)│
         └────────┬────────┘
                  │
    ┌─────────────┼─────────────┐
    ▼             ▼             ▼
┌───────┐  ┌──────────┐  ┌──────────┐
│ Поиск │  │Калькулятор│  │Веб-поиск │
│ в док.│  │          │  │         │
└───┬───┘  └────┬─────┘  └────┬────┘
    │           │              │
    └───────────┼──────────────┘
                ▼
         ┌─────────────────┐
         │  Оценка ответа  │
         │ (достаточно ли?) │
         └────────┬────────┘
                  │
        ┌─────────┼─────────┐
        ▼         ▼         ▼
    ┌───────┐ ┌───────┐ ┌───────┐
    │Ответ  │ │Повтор │ │Спросить│
    │польз. │ │поиска │ │польз.  │
    └───────┘ └───────┘ └───────┘
```

Главное преимущество: агент может **блуждать по графу** — возвращаться к предыдущим узлам, выбирать разные пути, останавливаться, когда цель достигнута.

### 1.3. Состояние — ядро агента

В LangGraph состояние — это объект, который передаётся между узлами. Обычно оно содержит:

- **Историю сообщений** (`messages`) — список всех сообщений в диалоге (аналогично памяти из Лекции 6.3).
- **Промежуточные данные** — результаты поиска, вычислений, флаги.
- **Метаданные** — текущий шаг, количество попыток, уверенность.

Вот как может выглядеть состояние для нашего агента:

```python
from typing import TypedDict, List, Annotated
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    # История диалога (автоматически добавляются новые сообщения)
    messages: Annotated[List[BaseMessage], add_messages]
    # Результаты поиска
    search_results: List[str]
    # Промежуточные вычисления
    calculations: str
    # Флаг, достаточно ли информации
    is_complete: bool
    # Количество попыток
    attempts: int
```

Аннотация `add_messages` — это специальный редьюсер, который говорит LangGraph: «когда узел добавляет новые сообщения в `messages`, не заменяй всё поле, а добавь их к существующему списку». Это ключевое поведение для поддержания истории диалога.

### 1.4. Установка LangGraph и связь с LangChain

LangGraph — это отдельная библиотека, но она тесно интегрирована с LangChain. Мы будем использовать те же компоненты: `ChatOllama` для LLM, `Chroma` для векторного поиска, `ChatPromptTemplate` для промптов.

**Установка:**

```bash
pip install langgraph
```

**Важно:** LangGraph требует Python 3.9+.

**Связь с LangChain:**
- LangGraph использует те же интерфейсы (`Runnable`, `BaseMessage`, `ChatPromptTemplate`).
- Мы можем вставлять цепочки LangChain внутрь узлов LangGraph.
- Всё, что мы изучили в Лекции 6.3, применимо и здесь.

**Проверка установки:**

```bash
python -c "import langgraph; print('✅ LangGraph готов')"
```

---

## Проектирование графа: что мы построим

Теперь, когда мы понимаем концепции, давайте спроектируем граф для нашего будущего агента. У нас будет несколько типов узлов:

1. **Маршрутизатор** — анализирует вопрос и историю, выбирает следующий узел.
2. **RAG-узел** — выполняет поиск в документах с помощью Chroma.
3. **Калькулятор** — выполняет математические операции (через Python `eval` или `numexpr`).
4. **Веб-поиск** — ищет в интернете через DuckDuckGo (используем `duckduckgo-search`).
5. **Время** — возвращает текущие дату и время.
6. **Генератор ответа** — формирует финальный ответ на основе собранной информации.
7. **Узел проверки** — оценивает, достаточно ли данных для ответа.

Переходы между узлами будут определяться маршрутизатором или результатами выполнения.

**Пример потока:**
1. Пользователь задаёт вопрос.
2. Маршрутизатор решает, что нужен поиск в документах.
3. RAG-узел выполняет поиск.
4. Маршрутизатор проверяет результат: если данных мало — направляет в веб-поиск.
5. Веб-поиск добавляет информацию.
6. Маршрутизатор решает, что данных достаточно → генерация ответа.
7. Ответ возвращается пользователю.

Этот граф даёт агенту **гибкость, адаптивность и возможность самоисправления** — качества, которых нет у линейных цепочек.



## Тема 2. Проектирование графа агента

Теперь, когда мы понимаем, зачем нужен LangGraph и как устроены графы состояний, давайте спроектируем нашего будущего агента. **Грамотный проект сэкономит нам кучу времени** — мы сразу увидим, какие узлы нужны, как они будут связаны и где возможны сложности.

В отличие от линейной цепочки, где путь данных предопределён, в графе мы должны **явно описать все возможные переходы**. Это требует чуть больше усилий на этапе проектирования, но даёт огромную гибкость на этапе реализации и расширения.

---

### 2.1. Узлы, которые мы создадим

Наш агент будет состоять из трёх основных узлов. Каждый узел — это функция, которая принимает текущее состояние, выполняет какое-то действие и возвращает обновлённое состояние.

#### 2.1.1. Узел `agent` — «мозг» агента

Этот узел — главный. Он принимает решение: что делать дальше. Внутри него мы вызываем LLM, которая анализирует текущее состояние (вопрос пользователя, историю диалога, результаты предыдущих действий) и решает:

- Если у неё достаточно информации для ответа — она генерирует финальный ответ.
- Если нужно что-то уточнить или найти — она вызывает один или несколько инструментов.

**Реализация:**

```python
def agent_node(state: AgentState) -> AgentState:
    """
    Узел-агент: вызывает LLM, которая решает, что делать дальше.
    Модель может вернуть либо ответ, либо запрос на вызов инструментов.
    """
    # Формируем сообщения для LLM (история + системный промпт)
    messages = state["messages"]
    
    # Вызываем LLM с привязанными инструментами
    # (инструменты мы подготовим в следующей теме)
    response = llm_with_tools.invoke(messages)
    
    # Добавляем ответ модели в историю
    return {
        "messages": [response]
    }
```

**Что здесь важно:**
- Мы используем `llm_with_tools` — специальную версию LLM, к которой привязаны инструменты. LangChain позволяет это сделать через `.bind_tools()`.
- Модель возвращает объект `AIMessage`, который может содержать либо текст ответа, либо запрос на вызов инструментов (поле `tool_calls`).
- Этот узел **не выполняет** инструменты, а только решает, какие инструменты нужны.

#### 2.1.2. Узел `tools` — исполнитель инструментов

Этот узел принимает запросы на вызов инструментов, выполняет их и добавляет результаты в состояние.

```python
from langgraph.prebuilt import ToolNode

# Создаём узел-исполнитель инструментов
tool_node = ToolNode(tools)
```

`ToolNode` — это готовая реализация узла от LangGraph, которая:
1. Извлекает из последнего сообщения `tool_calls`.
2. Вызывает соответствующие инструменты.
3. Возвращает результаты в виде сообщений `ToolMessage`.
4. Добавляет их в состояние.

**Как это работает:**

```python
# Пример вызова узла
result = tool_node.invoke(state)
# В state["messages"] добавляются ToolMessage с результатами
```

Если мы хотим кастомизировать поведение, можем написать свой узел, но для большинства случаев `ToolNode` достаточно.

#### 2.1.3. Узел `final_answer` — генерация финального ответа

Когда агент решил, что информации достаточно, и нужно ответить пользователю, мы переходим в этот узел. Здесь мы берём всю накопленную историю (включая результаты работы инструментов) и генерируем итоговый ответ.

**Важное отличие от `agent_node`:** здесь мы **не даём модели инструменты**. Это чистый вызов LLM для финального ответа.

```python
def final_answer_node(state: AgentState) -> AgentState:
    """
    Генерирует финальный ответ на основе всей накопленной истории.
    """
    messages = state["messages"]
    
    # Используем обычную LLM (без инструментов)
    response = llm.invoke(messages)
    
    # Добавляем финальный ответ в историю
    return {
        "messages": [response]
    }
```

**Почему отдельный узел?**
- Мы можем дать модели специальную инструкцию: «Теперь у тебя есть вся информация. Сформулируй чёткий и полный ответ пользователю».
- Мы можем ограничить длину ответа или применить другую температуру.
- Мы можем добавить пост-обработку (например, проверить, что ответ не содержит галлюцинаций).

---

### 2.2. Рёбра — логика переходов

Рёбра определяют, как агент перемещается между узлами. В нашем графе будет три типа переходов.

#### 2.2.1. `agent` → `tools`

Если модель в узле `agent` вернула запрос на вызов инструментов, мы переходим в узел `tools`.

```python
from langgraph.graph import StateGraph, END

# Добавляем условное ребро
graph.add_conditional_edges(
    "agent",
    should_continue,  # функция, которая решает, куда идти
    {
        "tools": "tools",
        "final": "final_answer"
    }
)
```

#### 2.2.2. `agent` → `final_answer`

Если модель не запросила инструменты (значит, она считает, что может ответить сразу), мы переходим в узел генерации финального ответа.

#### 2.2.3. `tools` → `agent`

После того как инструменты выполнились и их результаты добавлены в состояние, мы **возвращаемся в узел `agent`**, чтобы модель могла проанализировать результаты и решить: достаточно ли этого для ответа или нужно вызвать ещё инструменты.

Это создаёт **цикл**:

```
agent → tools → agent → tools → ... → final_answer
```

Агент может выполнить несколько итераций, пока не накопит достаточно информации или не достигнет лимита попыток.

---

### 2.3. Условные рёбра — функция `should_continue`

Условное ребро — это функция, которая смотрит на текущее состояние и возвращает имя следующего узла. Она реализует **логику принятия решений** агента.

```python
def should_continue(state: AgentState) -> str:
    """
    Определяет, куда идти дальше: вызывать инструменты или завершать.
    """
    messages = state["messages"]
    last_message = messages[-1]
    
    # Если последнее сообщение содержит запрос на вызов инструментов
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    
    # Иначе — переходим к финальному ответу
    return "final"
```

**Что здесь происходит:**

1. Мы смотрим на последнее сообщение в истории.
2. Если это `AIMessage` с полем `tool_calls` (не пустым) — значит, модель хочет что-то вызвать.
3. Возвращаем `"tools"` — идём в узел инструментов.
4. Если `tool_calls` нет — модель либо уже дала ответ, либо решила, что инструменты не нужны.
5. Возвращаем `"final"` — идём в узел `final_answer`.

Этот паттерн называется **ReAct (Reasoning + Acting)** — агент сначала **думает** (в узле `agent`), потом **действует** (в узле `tools`), потом снова **думает**, и так по кругу, пока не решит, что ответ готов.

---

### 2.4. Схема графа (текстовое описание)

Вот как выглядит наш граф в виде диаграммы:

```
                    ┌─────────────────────────────────────┐
                    │                                     │
                    ▼                                     │
           ┌───────────────────┐                        │
           │                   │                        │
           │  Узел: agent      │                        │
           │  ("Мозг")         │                        │
           │                   │                        │
           └─────────┬─────────┘                        │
                     │                                   │
                     │ Условное ребро                    │
                     │ (should_continue)                 │
                     │                                   │
        ┌────────────┼────────────┐                     │
        │            │            │                     │
        ▼            ▼            ▼                     │
  ┌───────────┐ ┌───────────┐ ┌───────────┐            │
  │           │ │           │ │           │            │
  │ tools     │ │ final     │ │ (останов) │            │
  │ (выпол-   │ │ answer    │ │           │            │
  │ нение)    │ │ (финаль-  │ │           │            │
  │           │ │ ный ответ)│ │           │            │
  └─────┬─────┘ └───────────┘ └───────────┘            │
        │              │                                 │
        │              ▼                                 │
        │        ┌───────────┐                          │
        │        │   END     │                          │
        │        │ (завер-   │                          │
        │        │ шение)    │                          │
        │        └───────────┘                          │
        │                                               │
        └────────────────────────────────────────────────┘
```

**Что мы видим:**

1. Агент начинается с узла `agent`.
2. После выполнения `agent` вызывается функция `should_continue`.
3. Если есть `tool_calls` → переход в `tools`.
4. После `tools` — возврат в `agent` (цикл).
5. Если `tool_calls` нет → переход в `final_answer`.
6. После `final_answer` — остановка (END).

**Количество итераций** ограничено либо самим агентом (он решает, что достаточно), либо мы можем установить максимальное число шагов (например, 5), чтобы избежать бесконечных циклов.

---

### 2.5. Преимущества такой архитектуры

| Аспект | Линейная цепочка (LCEL) | Граф с циклом (LangGraph) |
|--------|-------------------------|---------------------------|
| **Ветвление** | Нет | Да (условные рёбра) |
| **Циклы** | Нет | Да (tools → agent) |
| **Гибкость** | Низкая (путь фиксирован) | Высокая (агент сам выбирает путь) |
| **Масштабируемость** | Добавление нового шага = правка всей цепочки | Добавление нового узла = простое расширение графа |
| **Прозрачность** | Сложно отлаживать | Каждый шаг логируется в состоянии |

---

## Плавный переход к следующей теме

Узлы и рёбра готовы на бумаге. Мы знаем, что наш агент будет состоять из трёх узлов, одного условного ребра и двух типов переходов. Но чтобы агент мог **действовать**, ему нужны **инструменты** — те самые «руки», которыми он будет выполнять задачи.

В следующей теме мы подготовим:
- **Поиск по документам** — ретривер из Лекции 6.3.
- **Калькулятор** — для математических операций.
- **Веб-поиск** — для поиска в интернете.
- **Текущее время** — простой, но полезный инструмент.

Каждый инструмент будет представлен как функция с описанием, которое модель будет использовать для выбора. Мы привяжем эти инструменты к LLM через `.bind_tools()`, и тогда наш агент сможет **осознанно** решать, что и когда вызывать.



## Тема 3. Инструменты в LangGraph (декоратор @tool)

Мы спроектировали граф и определили узлы. Теперь настало время дать нашему агенту **«руки»** — инструменты, которыми он сможет пользоваться. Инструменты — это функции, которые агент может вызывать для выполнения конкретных действий: поиска в документах, вычислений, получения времени, поиска в интернете. В LangChain и LangGraph создание инструментов максимально упрощено с помощью декоратора `@tool`.

**Почему декоратор `@tool`?**

- Он автоматически создаёт из функции объект `StructuredTool`.
- Извлекает название и docstring функции (становится описанием для модели).
- Автоматически генерирует Pydantic-схему для параметров (типы, описания).
- Делает инструмент готовым к передаче в LLM через `.bind_tools()`.

Всё это позволяет модели понимать, **что** делает инструмент, **когда** его использовать и **какие параметры** ему нужны.

---

### 3.1. Создание инструментов

Мы создадим четыре инструмента, которые будем использовать в нашем агенте:

1. **`search_docs(query)`** — поиск по документам в Chroma.
2. **`calculate(expression)`** — выполнение математических вычислений.
3. **`get_current_time()`** — получение текущей даты и времени.
4. **`web_search(query)`** — поиск в интернете через DuckDuckGo.

#### 3.1.1. Инструмент `search_docs` — поиск в документах

Этот инструмент использует ретривер из Лекции 6.3. Он принимает текстовый запрос и возвращает три самых релевантных фрагмента из базы знаний.

```python
import re
from langchain.tools import tool

@tool
def search_docs(query: str) -> str:
    """
    Ищет информацию в локальной базе знаний (документы компании).
    Используй этот инструмент, когда вопрос касается:
    - продуктов компании (ProjectFlow)
    - финансовых отчётов
    - истории компании
    - внутренних процессов и документации
    Аргумент: query — поисковый запрос (строка).
    Возвращает: найденные фрагменты текста, разделённые тремя дефисами.
    """
    try:
        # Используем ретривер из глобальной области или передаём через замыкание
        docs = retriever.invoke(query)
        if not docs:
            return "По вашему запросу ничего не найдено."
        
        results = []
        for doc in docs:
            source = doc.metadata.get("file_name", "неизвестный источник")
            results.append(f"Источник: {source}\n{doc.page_content}")
        
        return "\n\n---\n\n".join(results)
    except Exception as e:
        return f"Ошибка при поиске: {str(e)}"
```

**Что здесь важно:**
- Docstring описывает, когда использовать инструмент — модель будет использовать эту информацию для принятия решений.
- `retriever` должен быть доступен в области видимости (мы можем передать его через замыкание или глобальную переменную).
- Возвращаемое значение — строка, которая будет добавлена в историю как `ToolMessage`.

#### 3.1.2. Инструмент `calculate` — математический калькулятор

Этот инструмент выполняет арифметические операции. Мы используем безопасный `eval` с ограниченным набором функций.

```python
import math

@tool
def calculate(expression: str) -> str:
    """
    Выполняет математические вычисления. Поддерживает операции: +, -, *, /, **, %, а также функции: sqrt, sin, cos, tan, log, log10, pi, e.
    Используй этот инструмент, когда пользователь просит что-то посчитать: сумму, процент, корень, тригонометрию и т.д.
    Аргумент: expression — математическое выражение (строка), например '2+2' или 'sqrt(16)'.
    Возвращает: результат вычисления или сообщение об ошибке.
    """
    try:
        # Разрешаем только безопасные функции и константы
        safe_dict = {
            'sqrt': math.sqrt,
            'sin': math.sin,
            'cos': math.cos,
            'tan': math.tan,
            'log': math.log,
            'log10': math.log10,
            'pi': math.pi,
            'e': math.e,
            'abs': abs,
            'round': round,
            'sum': sum,
            'min': min,
            'max': max,
        }
        # Удаляем потенциально опасные символы
        cleaned = re.sub(r'[^0-9+\-*/%().,sqrt sincostanlogpi eabsround]', '', expression.lower())
        result = eval(cleaned, {"__builtins__": {}}, safe_dict)
        return f"Результат вычисления: {result}"
    except Exception as e:
        return f"Ошибка в вычислении: {str(e)}"
```

**Обратите внимание:**
- Мы используем `eval()` с ограничениями (пустой `__builtins__` и только разрешённые функции).
- Это **не полностью безопасно**, но для демонстрации подходит. В продакшене лучше использовать `numexpr` или парсеры.

#### 3.1.3. Инструмент `get_current_time` — текущее время

Самый простой инструмент, который не требует параметров.

```python
from datetime import datetime

@tool
def get_current_time() -> str:
    """
    Возвращает текущие дату и время в формате 'ГГГГ-ММ-ДД ЧЧ:ММ:СС'.
    Используй этот инструмент, когда пользователь спрашивает о текущем времени, дате, дне недели.
    """
    now = datetime.now()
    return f"Текущее время: {now.strftime('%Y-%m-%d %H:%M:%S')}"
```

#### 3.1.4. Инструмент `web_search` — поиск в интернете

Для этого инструмента нам понадобится библиотека `duckduckgo-search`.

**Установка:**

```bash
pip install duckduckgo-search
```

```python
from duckduckgo_search import DDGS

@tool
def web_search(query: str) -> str:
    """
    Выполняет поиск в интернете через DuckDuckGo.
    Используй этот инструмент, когда:
    - В локальной базе знаний нет информации.
    - Нужны свежие данные или новости.
    - Вопрос касается общих знаний, которых нет в документах компании.
    Аргумент: query — поисковый запрос (строка).
    Возвращает: список результатов (заголовки и ссылки).
    """
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
            if not results:
                return "По вашему запросу ничего не найдено."
            
            formatted = []
            for i, r in enumerate(results, 1):
                formatted.append(f"{i}. {r['title']}\n   {r['body'][:200]}...\n   Источник: {r['href']}")
            
            return "\n\n".join(formatted)
    except Exception as e:
        return f"Ошибка при веб-поиске: {str(e)}"
```

---

### 3.2. Оборачивание функций в `@tool`

Декоратор `@tool` из `langchain.tools` автоматически:

1. Превращает функцию в объект `StructuredTool`.
2. Извлекает имя функции (или можно задать своё через `name`).
3. Извлекает docstring (описание для модели).
4. Анализирует аннотации типов и генерирует Pydantic-схему для параметров.
5. Добавляет метод `invoke()` для вызова.

**Пример настройки имени и описания вручную:**

```python
@tool(name="search_docs", description="Ищет в документах компании")
def search_docs(query: str) -> str:
    # ...
```

Но обычно достаточно хорошего docstring и аннотаций.

**Где хранить инструменты?**

```python
tools = [search_docs, calculate, get_current_time, web_search]
```

Мы можем добавлять и удалять инструменты, просто меняя этот список.

---

### 3.3. Привязка инструментов к модели через `.bind_tools()`

В LangChain модели не знают об инструментах по умолчанию. Чтобы дать им доступ, мы используем метод `.bind_tools()`.

```python
from langchain_ollama import ChatOllama

# Обычная модель
llm = ChatOllama(model="qwen2.5:3b", temperature=0.0)

# Модель с привязанными инструментами
llm_with_tools = llm.bind_tools(tools)
```

**Что делает `.bind_tools()`?**

1. Модифицирует промпт модели, добавляя информацию о доступных инструментах (их названия, описания, схемы параметров).
2. Настраивает модель на возврат структурированного ответа с полем `tool_calls`.
3. Если модель решает, что нужен инструмент, она возвращает `AIMessage` с `tool_calls` вместо обычного текста.

**Важно:** Не все модели поддерживают `bind_tools()` одинаково. Для Ollama это работает через специальный формат промпта. Если возникают проблемы, можно использовать `tools` в `ChatPromptTemplate` (но это сложнее).

---

### 3.4. Демонстрация: вызов LLM с инструментами

Давайте протестируем, как модель «видит» инструменты и принимает решения.

**Создадим файл `test_tools.py`:**

```python
from langchain_ollama import ChatOllama
from langchain.tools import tool
import math
import re
from datetime import datetime

# Определяем инструменты (упрощённая версия для демонстрации)
@tool
def calculate(expression: str) -> str:
    """Выполняет математические вычисления."""
    try:
        result = eval(expression, {"__builtins__": {}}, {"sqrt": math.sqrt, "pi": math.pi})
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка: {e}"

@tool
def get_current_time() -> str:
    """Возвращает текущее время."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

tools = [calculate, get_current_time]

# Создаём модель с инструментами
llm = ChatOllama(model="qwen2.5:3b", temperature=0.0)
llm_with_tools = llm.bind_tools(tools)

# Тестируем
questions = [
    "Сколько будет 123 * 456?",
    "Который сейчас час?",
    "Как тебя зовут?"  # не требует инструментов
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Вопрос: {q}")
    print('-'*60)
    
    response = llm_with_tools.invoke(q)
    
    # Проверяем, вызвала ли модель инструмент
    if hasattr(response, 'tool_calls') and response.tool_calls:
        print(f"🔧 Модель вызвала инструмент: {response.tool_calls[0]['name']}")
        print(f"   Параметры: {response.tool_calls[0]['args']}")
    else:
        print(f"💬 Ответ модели: {response.content}")
```

**Ожидаемый вывод:**

```
============================================================
Вопрос: Сколько будет 123 * 456?
------------------------------------------------------------
🔧 Модель вызвала инструмент: calculate
   Параметры: {'expression': '123*456'}

============================================================
Вопрос: Который сейчас час?
------------------------------------------------------------
🔧 Модель вызвала инструмент: get_current_time
   Параметры: {}

============================================================
Вопрос: Как тебя зовут?
------------------------------------------------------------
💬 Ответ модели: Меня зовут помощник. А как зовут тебя?
```

**Что мы видим:**

1. Для математического вопроса модель выбрала инструмент `calculate` и передала выражение.
2. Для вопроса о времени — инструмент `get_current_time`.
3. Для общего вопроса — ответила без инструментов.

**Структура `tool_calls`:**

```python
[
    {
        'name': 'calculate',
        'args': {'expression': '123*456'},
        'id': 'call_abc123'
    }
]
```

Мы можем взять это имя, найти соответствующий инструмент и выполнить его.

---

### 3.5. Что дальше?

Теперь у нашего агента есть «руки» — инструменты для выполнения действий. Мы знаем, как:

- Создавать инструменты с помощью `@tool`.
- Привязывать их к модели через `.bind_tools()`.
- Распознавать в ответе модели запросы на вызов инструментов (`tool_calls`).

Остаётся самое интересное: **собрать мозг и тело в работающий граф**.

В следующей теме мы:
1. Создадим функцию-агент, которая будет использовать `llm_with_tools`.
2. Определим узел `tools` с помощью `ToolNode`.
3. Свяжем всё в единый граф с помощью `StateGraph`.
4. Добавим цикл, чтобы агент мог выполнять несколько шагов.

**Пока что наш агент умеет только выбирать инструмент. В следующей теме он сможет выполнять их и анализировать результат.** Оставайтесь с нами!


## Тема 4. Реализация цикла «агент-инструменты-агент» (скрипт `agent_graph.py`)

Это центральная часть лекции. Мы превращаем статичные узлы в **живой цикл**, где агент может многократно вызывать инструменты, анализировать результаты и принимать решения. Именно этот паттерн делает агента «настоящим» — способным к самонаправленному исследованию и решению сложных задач.

До этого момента у нас были:
- **Инструменты** — готовые функции (поиск, калькулятор, время, веб-поиск).
- **LLM с привязанными инструментами** — модель знает, что она может вызывать.
- **План графа** — три узла и условное ребро.

Теперь мы соединим всё в единую систему, которая будет работать по принципу:

1. **Агент** (LLM) получает вопрос и историю → решает, что делать.
2. Если нужны инструменты → вызывает их.
3. **Инструменты** выполняются → результаты добавляются в историю.
4. Управление возвращается к **Агенту**.
5. Цикл повторяется, пока агент не решит, что готов ответить.
6. **Финальный ответ** генерируется и возвращается пользователю.

Этот цикл называется **ReAct (Reasoning + Acting)** — агент чередует «размышление» и «действие».

---

### 4.1. Определение состояния с TypedDict и reducer `add_messages`

Состояние — это «память» агента, которая передаётся между узлами. В LangGraph состояние определяется как TypedDict. Ключевой элемент — поле `messages`, которое содержит всю историю диалога, включая сообщения пользователя, ответы модели и результаты инструментов.

```python
from typing import Annotated, List, TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    """
    Состояние агента.
    messages: список всех сообщений в диалоге.
    add_messages — reducer, который дополняет список, а не заменяет.
    """
    messages: Annotated[List[BaseMessage], add_messages]
```

**Что такое `add_messages`?**

В обычном словаре, если узел возвращает `{"messages": [new_msg]}`, он заменит всё поле `messages`. Но с `add_messages` LangGraph **дополняет** существующий список новыми сообщениями. Это критически важно для сохранения истории диалога.

Без этого reducer каждый узел «забывал» бы предыдущие сообщения и агент не мог бы вести связный диалог.

---

### 4.2. Узел `agent` — «мозг» агента

Узел `agent` — это функция, которая вызывает LLM с привязанными инструментами и возвращает ответ модели.

```python
from langchain_ollama import ChatOllama

# Модель с инструментами (создаём один раз, вне узла)
llm = ChatOllama(model="qwen2.5:3b", temperature=0.0)
llm_with_tools = llm.bind_tools(tools)

def agent_node(state: AgentState) -> AgentState:
    """
    Узел-агент: вызывает LLM с текущей историей и возвращает ответ.
    """
    # Извлекаем историю сообщений
    messages = state["messages"]
    
    # Вызываем модель с привязанными инструментами
    response = llm_with_tools.invoke(messages)
    
    # Возвращаем новое состояние (сообщение модели добавляется в историю)
    return {"messages": [response]}
```

**Что здесь важно:**
- Мы используем `llm_with_tools` — модель, которая знает об инструментах.
- Возвращаемый словарь будет автоматически объединён с текущим состоянием через `add_messages`.
- Мы не вызываем инструменты внутри этого узла — только решаем, нужны ли они.

---

### 4.3. Функция `should_continue` — условное ребро

Эта функция определяет, куда идти дальше: вызывать инструменты или сразу переходить к финальному ответу.

```python
def should_continue(state: AgentState) -> str:
    """
    Определяет следующий узел на основе последнего сообщения.
    """
    messages = state["messages"]
    last_message = messages[-1]
    
    # Если последнее сообщение содержит запрос на вызов инструментов
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"  # идём в узел инструментов
    
    # Иначе — генерируем финальный ответ
    return "final_answer"
```

**Логика:**
- Модель может вернуть `AIMessage` с полем `tool_calls` (список запросов).
- Если такой список не пуст — агент хочет что-то сделать → переход в `tools`.
- Если `tool_calls` пуст или отсутствует — агент готов ответить → переход в `final_answer`.

---

### 4.4. Узел `tools` — выполнение инструментов

Этот узел принимает запросы на вызов инструментов, выполняет их и добавляет результаты в состояние.

**Вариант А: Использование готового `ToolNode` (рекомендуется)**

LangGraph предоставляет готовый узел `ToolNode`, который автоматически:
1. Извлекает `tool_calls` из последнего сообщения.
2. Вызывает соответствующие инструменты по имени.
3. Возвращает `ToolMessage` с результатами.

```python
from langgraph.prebuilt import ToolNode

# Создаём узел из списка инструментов
tool_node = ToolNode(tools)
```

**Вариант Б: Ручная реализация (для понимания)**

Если мы хотим понять, как это работает внутри, можно написать свой узел:

```python
def manual_tool_node(state: AgentState) -> AgentState:
    """
    Ручная реализация узла инструментов.
    """
    messages = state["messages"]
    last_message = messages[-1]
    
    # Проверяем наличие tool_calls
    if not hasattr(last_message, "tool_calls") or not last_message.tool_calls:
        return {"messages": []}  # ничего не делаем
    
    tool_messages = []
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_id = tool_call["id"]
        
        # Находим нужный инструмент
        tool = next((t for t in tools if t.name == tool_name), None)
        if tool:
            try:
                result = tool.invoke(tool_args)
                tool_messages.append({
                    "role": "tool",
                    "content": result,
                    "tool_call_id": tool_id
                })
            except Exception as e:
                tool_messages.append({
                    "role": "tool",
                    "content": f"Ошибка: {str(e)}",
                    "tool_call_id": tool_id
                })
        else:
            tool_messages.append({
                "role": "tool",
                "content": f"Инструмент '{tool_name}' не найден",
                "tool_call_id": tool_id
            })
    
    return {"messages": tool_messages}
```

**Для простоты будем использовать готовый `ToolNode`.**

---

### 4.5. Узел `final_answer` — генерация финального ответа

Когда агент решил, что информации достаточно, мы переходим в этот узел. Здесь мы вызываем LLM без инструментов, чтобы сформулировать ответ пользователю.

```python
def final_answer_node(state: AgentState) -> AgentState:
    """
    Генерирует финальный ответ на основе всей истории.
    """
    messages = state["messages"]
    
    # Используем обычную LLM (без инструментов)
    response = llm.invoke(messages)
    
    return {"messages": [response]}
```

**Зачем отдельный узел?**
- Мы можем дать модели специальную инструкцию: «Теперь у тебя есть вся информация. Сформулируй чёткий и полный ответ».
- Мы можем использовать другую температуру или модель для финального ответа.
- Можно добавить пост-обработку (проверка, форматирование).

---

### 4.6. Сборка графа

Теперь собираем граф из всех узлов и рёбер.

```python
from langgraph.graph import StateGraph, END

# 1. Создаём граф с состоянием
builder = StateGraph(AgentState)

# 2. Добавляем узлы
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)  # готовый ToolNode
builder.add_node("final_answer", final_answer_node)

# 3. Устанавливаем точку входа
builder.set_entry_point("agent")

# 4. Добавляем условное ребро из agent
builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "final_answer": "final_answer"
    }
)

# 5. Добавляем ребро из tools обратно в agent (цикл)
builder.add_edge("tools", "agent")

# 6. Добавляем ребро из final_answer в END (завершение)
builder.add_edge("final_answer", END)

# 7. Компилируем граф
graph = builder.compile()
```

**Схема графа в коде:**

```
agent
  │
  ├─ (should_continue: tool_calls есть) → tools → agent (цикл)
  │
  └─ (should_continue: tool_calls нет) → final_answer → END
```

---

### 4.7. Тестовый запуск и защита от бесконечного цикла

При запуске графа нам нужно ограничить количество итераций, чтобы агент не зациклился. В LangGraph это делается через `config` с параметром `recursion_limit`.

```python
# Тестовые вопросы
questions = [
    "Сколько будет 2+2?",
    "Какая сейчас погода в Москве?",
    "Что такое ProjectFlow?"
]

# Конфигурация с лимитом итераций
config = {"recursion_limit": 10}

for question in questions:
    print(f"\n{'='*60}")
    print(f"Вопрос: {question}")
    print('-'*60)
    
    # Начальное состояние
    initial_state = {
        "messages": [("user", question)]
    }
    
    # Запускаем граф
    result = graph.invoke(initial_state, config=config)
    
    # Извлекаем последний ответ
    last_message = result["messages"][-1]
    print(f"Ответ: {last_message.content}")
```

**Ожидаемый вывод (схематично):**

```
============================================================
Вопрос: Сколько будет 2+2?
------------------------------------------------------------
🔧 Агент вызвал инструмент: calculate('2+2')
📊 Результат: 4
🔧 Агент вызвал инструмент: calculate('2+2')
💬 Финальный ответ: 4

============================================================
Вопрос: Какая сейчас погода в Москве?
------------------------------------------------------------
🔧 Агент вызвал инструмент: web_search('погода в Москве')
📊 Результат: [Результаты поиска]
💬 Финальный ответ: По данным поиска, в Москве сейчас +15°C, облачно.

============================================================
Вопрос: Что такое ProjectFlow?
------------------------------------------------------------
🔧 Агент вызвал инструмент: search_docs('ProjectFlow')
📊 Результат: [Документы компании]
💬 Финальный ответ: ProjectFlow — это облачная платформа для управления задачами...
```

**Что происходит внутри:**

1. **Вопрос 1:** Агент вызывает `calculate(2+2)`, получает результат, затем решает, что этого достаточно, и переходит к финальному ответу.

2. **Вопрос 2:** Агент не находит информации в документах, решает использовать веб-поиск, получает результаты, анализирует их и формулирует ответ.

3. **Вопрос 3:** Агент сразу находит информацию в документах через `search_docs` и отвечает.

**Защита от бесконечного цикла:**

- `recursion_limit=10` ограничивает общее количество шагов (узлов).
- Если агент не завершится за 10 шагов, LangGraph выбросит исключение `GraphRecursionError`.
- В реальном приложении можно установить лимит в 20–50 шагов и обрабатывать ошибку.

---

### 4.8. Полный код для тестирования (скрипт `agent_graph.py`)

Сохраните следующий код в файл **`agent_graph.py`**. Это основной скрипт лекции, который содержит все компоненты: инструменты, состояние, узлы, граф и тестовый запуск.

```python
"""
agent_graph.py - Полноценный агент на LangGraph с инструментами
Лекция 6.4: Агент на LangGraph

Возможности:
- Цикл «агент → инструменты → агент» (ReAct)
- Инструменты: калькулятор, время, веб-поиск, поиск в документах
- Защита от бесконечного цикла (recursion_limit)
- Состояние с историей сообщений
"""

import math
import re
from datetime import datetime
from typing import Annotated, List, TypedDict

from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# ============================================================================
# 1. ИНСТРУМЕНТЫ
# ============================================================================

@tool
def calculate(expression: str) -> str:
    """
    Выполняет математические вычисления.
    Поддерживает: +, -, *, /, **, %, sqrt, sin, cos, tan, pi, e.
    Используй этот инструмент, когда нужно что-то посчитать.
    """
    try:
        safe_dict = {
            'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos, 'tan': math.tan,
            'pi': math.pi, 'e': math.e, 'abs': abs, 'round': round
        }
        # Разрешаем только безопасные символы и функции
        cleaned = re.sub(r'[^0-9+\-*/%().,sqrt sincostanlogpi eabsround]', '', expression.lower())
        result = eval(cleaned, {"__builtins__": {}}, safe_dict)
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {e}"

@tool
def get_current_time() -> str:
    """
    Возвращает текущие дату и время в формате ГГГГ-ММ-ДД ЧЧ:ММ:СС.
    Используй этот инструмент, когда пользователь спрашивает о времени, дате или дне недели.
    """
    now = datetime.now()
    return now.strftime("%Y-%m-%d %H:%M:%S")

@tool
def web_search(query: str) -> str:
    """
    Выполняет поиск в интернете через DuckDuckGo.
    Используй, когда нужны свежие данные, новости или информация из интернета.
    """
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
            if not results:
                return "По вашему запросу ничего не найдено."
            formatted = []
            for i, r in enumerate(results, 1):
                formatted.append(f"{i}. {r['title']}\n   {r['body'][:150]}...\n   Источник: {r['href']}")
            return "\n\n".join(formatted)
    except ImportError:
        return ("Библиотека duckduckgo-search не установлена. "
                "Установите: pip install duckduckgo-search")
    except Exception as e:
        return f"Ошибка веб-поиска: {e}"

@tool
def search_docs(query: str) -> str:
    """
    Ищет информацию в локальной базе знаний (документы компании).
    Используй для вопросов о продуктах, финансовых отчётах, истории компании.
    """
    # В реальном проекте здесь должен быть retriever из Лекции 6.3
    # Для демонстрации возвращаем заглушку
    return (f"🔍 Поиск в документах по запросу: '{query}'\n"
            f"(Здесь будет реальный поиск из Chroma при подключении ретривера)")

# Список всех инструментов
tools = [calculate, get_current_time, web_search, search_docs]

# ============================================================================
# 2. LLM С ИНСТРУМЕНТАМИ
# ============================================================================

# СИСТЕМНЫЙ ПРОМПТ — объясняет агенту, как использовать инструменты
SYSTEM_PROMPT = """Ты — интеллектуальный агент с доступом к инструментам.

Инструменты:
- calculate: выполняет математические вычисления (пример: '2+2', 'sqrt(16)').
- get_current_time: возвращает текущие дату и время.
- web_search: ищет информацию в интернете (используй для свежих данных, новостей).
- search_docs: ищет в локальной базе знаний (продукты компании, отчёты, история).

Правила работы:
1. Если вопрос требует вычислений — используй calculate.
2. Если вопрос о времени/дате — используй get_current_time.
3. Если вопрос требует свежей информации из интернета — используй web_search.
4. Если вопрос о продуктах компании или внутренних документах — используй search_docs.
5. После получения результата от инструмента, сформулируй чёткий и полный ответ пользователю.
6. Не придумывай информацию, если её нет в результатах инструментов.
7. Если результат инструмента содержит ошибку — сообщи об этом пользователю.
"""

llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0.0,
    num_predict=512
)

# Привязываем инструменты к модели
llm_with_tools = llm.bind_tools(tools)

# ============================================================================
# 3. СОСТОЯНИЕ
# ============================================================================

class AgentState(TypedDict):
    """Состояние агента: список сообщений с автоматическим дополнением."""
    messages: Annotated[List[BaseMessage], add_messages]

# ============================================================================
# 4. УЗЛЫ ГРАФА
# ============================================================================

def agent_node(state: AgentState) -> AgentState:
    """
    Узел-агент: вызывает LLM с привязанными инструментами.
    Модель решает, нужны ли инструменты или можно ответить сразу.
    """
    messages = state["messages"]
    
    # Добавляем системное сообщение, если его ещё нет
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def should_continue(state: AgentState) -> str:
    """
    Функция условного ребра.
    Определяет, нужно ли вызывать инструменты или переходить к финальному ответу.
    """
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"      # есть запрос на вызов инструментов
    return "final_answer"   # инструменты не нужны

def final_answer_node(state: AgentState) -> AgentState:
    """
    Узел финального ответа: генерирует ответ без инструментов.
    Использует всю накопленную историю для формулировки ответа.
    """
    messages = state["messages"]
    
    # Добавляем системное сообщение для финального ответа, если его нет
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    
    # Для финального ответа используем увеличенный лимит токенов
    llm_final = ChatOllama(
        model="qwen2.5:3b",
        temperature=0.0,
        num_predict=1024
    )
    
    response = llm_final.invoke(messages)
    
    # Проверяем, не пустой ли ответ
    if not response.content or len(response.content.strip()) < 5:
        # Если ответ пустой, просим модель ответить более развёрнуто
        fallback_prompt = """Ты должен дать полный и чёткий ответ на вопрос пользователя, используя всю доступную информацию.
Если ты использовал инструменты, включи их результаты в ответ.
Ответь прямо, без лишних пояснений и без "я не знаю"."""
        response = llm_final.invoke(messages + [HumanMessage(content=fallback_prompt)])
    
    return {"messages": [response]}

# Узел для выполнения инструментов (готовый из LangGraph)
tool_node = ToolNode(tools)

# ============================================================================
# 5. СБОРКА ГРАФА
# ============================================================================

# Создаём граф с состоянием AgentState
builder = StateGraph(AgentState)

# Добавляем узлы
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_node("final_answer", final_answer_node)

# Точка входа — узел agent
builder.set_entry_point("agent")

# Условное ребро из agent: если есть tool_calls → tools, иначе → final_answer
builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "final_answer": "final_answer"
    }
)

# Ребро из tools обратно в agent (цикл)
builder.add_edge("tools", "agent")

# Ребро из final_answer в END (завершение)
builder.add_edge("final_answer", END)

# Компилируем граф
graph = builder.compile()

# ============================================================================
# 6. ТЕСТИРОВАНИЕ
# ============================================================================

if __name__ == "__main__":
    # Настройка: ограничение числа итераций (защита от бесконечного цикла)
    config = {"recursion_limit": 10}
    
    # Тестовые вопросы
    questions = [
        "Сколько будет 2+2?",
        "Который сейчас час?",
        "Что такое LangGraph?",
        "Какой сегодня день недели?"
    ]
    
    print("=" * 60)
    print("🤖 ТЕСТИРОВАНИЕ АГЕНТА НА LANGGRAPH")
    print("=" * 60)
    
    for q in questions:
        print(f"\n{'=' * 60}")
        print(f"📝 Вопрос: {q}")
        print('-' * 60)
        
        try:
            # Запускаем граф
            result = graph.invoke(
                {"messages": [("user", q)]},
                config=config
            )
            
            # Выводим все шаги для наглядности
            print("📋 Шаги выполнения:")
            for i, msg in enumerate(result["messages"]):
                msg_type = type(msg).__name__
                if msg_type == "HumanMessage":
                    print(f"  {i+1}. 👤 Пользователь: {msg.content}")
                elif msg_type == "AIMessage":
                    if hasattr(msg, "tool_calls") and msg.tool_calls:
                        tools_called = ", ".join([tc["name"] for tc in msg.tool_calls])
                        print(f"  {i+1}. 🤖 Агент вызвал: {tools_called}")
                    else:
                        print(f"  {i+1}. 🤖 Агент: {msg.content[:100]}...")
                elif msg_type == "ToolMessage":
                    print(f"  {i+1}. 🔧 Инструмент: {msg.content[:100]}...")
            
            # Извлекаем последнее сообщение (финальный ответ)
            last_msg = result["messages"][-1]
            print(f"\n✅ Финальный ответ: {last_msg.content}")
            print(f"📊 Всего шагов: {len(result['messages'])}")
            
        except Exception as e:
            print(f"❌ Ошибка: {e}")
```

---

### 4.9. Как запустить скрипт

1. **Сохраните файл** как `agent_graph.py` в корне вашего проекта.

2. **Установите необходимые библиотеки** (если ещё не установлены):
   ```bash
   pip install langgraph duckduckgo-search
   ```

3. **Убедитесь, что Ollama запущен** и модель `qwen2.5:3b` загружена:
   ```bash
   ollama serve
   ollama pull qwen2.5:3b
   ```

4. **Запустите скрипт**:
   ```bash
   python agent_graph.py
   ```

5. **Ожидаемый улучшенный вывод** (после добавления системного промпта):

   ```
   ============================================================
   🤖 ТЕСТИРОВАНИЕ АГЕНТА НА LANGGRAPH
   ============================================================
   
   ============================================================
   📝 Вопрос: Сколько будет 2+2?
   ------------------------------------------------------------
   📋 Шаги выполнения:
     1. 👤 Пользователь: Сколько будет 2+2?
     2. 🤖 Агент вызвал: calculate
     3. 🔧 Инструмент: Результат: 4...
     4. 🤖 Агент: Результат вычисления: 4
   
   ✅ Финальный ответ: Результат вычисления: 4
   📊 Всего шагов: 4
   
   ============================================================
   📝 Вопрос: Который сейчас час?
   ------------------------------------------------------------
   📋 Шаги выполнения:
     1. 👤 Пользователь: Который сейчас час?
     2. 🤖 Агент вызвал: get_current_time
     3. 🔧 Инструмент: 2026-08-06 15:30:45
     4. 🤖 Агент: Текущее время: 2026-08-06 15:30:45
   
   ✅ Финальный ответ: Текущее время: 2026-08-06 15:30:45
   📊 Всего шагов: 4
   
   ============================================================
   📝 Вопрос: Что такое LangGraph?
   ------------------------------------------------------------
   📋 Шаги выполнения:
     1. 👤 Пользователь: Что такое LangGraph?
     2. 🤖 Агент: LangGraph — это библиотека от создателей LangChain...
   
   ✅ Финальный ответ: LangGraph — это библиотека от создателей LangChain...
   📊 Всего шагов: 2
   ```

---

### 4.10. Подключение реального поиска по документам

Чтобы агент мог искать в реальных документах, замените заглушку `search_docs` на реальный ретривер из Лекции 6.3. Добавьте в начало файла:

```python
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Загрузка существующей базы Chroma
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding_model,
    collection_name="rag_docs"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
```

А затем замените функцию `search_docs`:

```python
@tool
def search_docs(query: str) -> str:
    """Ищет информацию в локальной базе знаний (документы компании)."""
    try:
        docs = retriever.invoke(query)
        if not docs:
            return "По вашему запросу ничего не найдено в документах."
        results = []
        for doc in docs:
            source = doc.metadata.get("file_name", "неизвестный источник")
            results.append(f"Источник: {source}\n{doc.page_content}")
        return "\n\n---\n\n".join(results)
    except Exception as e:
        return f"Ошибка поиска в документах: {e}"
```

---

## Краткий итог Тема 4

- Мы реализовали **цикл «агент → инструменты → агент»** с использованием LangGraph в скрипте `agent_graph.py`.

- **Состояние** определяется через TypedDict с `add_messages`, что автоматически сохраняет историю диалога.

- **Узел `agent`** вызывает LLM с привязанными инструментами и решает, что делать дальше.

- **Функция `should_continue`** определяет, нужны ли инструменты или финальный ответ.

- **Узел `tools`** (через `ToolNode`) выполняет вызванные инструменты и возвращает результаты.

- **Узел `final_answer`** генерирует итоговый ответ без инструментов.

- **Граф** собран с циклическим ребром `tools → agent`, что позволяет агенту выполнять несколько шагов.

- **Защита от бесконечного цикла** реализована через `recursion_limit`.

- **Скрипт `agent_graph.py`** содержит полный код и готов к запуску.

---

**Теперь у агента есть «мозг» (LLM) и «руки» (инструменты), соединённые в цикл. Он может самостоятельно решать, когда и какие инструменты вызывать, и делать это многократно, пока не накопит достаточно информации для ответа.**

В следующей теме мы **упакуем всё в финальный класс `LangGraphAgent`**, добавим логирование и сделаем его готовым к использованию в реальных проектах. Оставайтесь с нами!


## Тема 5. Добавление памяти (сохранение состояния между запросами) — скрипт `agent_with_memory.py`

Мы построили агента, который умеет вызывать инструменты и анализировать результаты. Но есть одна проблема: **он не помнит предыдущие вопросы**. Каждый вызов `graph.invoke()` начинается с чистого состояния — история диалога не сохраняется между запросами. Это делает агента бесполезным в диалоговом режиме.

В Лекции 6.3 мы решали эту проблему вручную — хранили список сообщений в `InMemoryChatMessageHistory` и оборачивали цепочку в `RunnableWithMessageHistory`. В LangGraph это делается ещё проще и элегантнее: через **чекпоинты (checkpoints)**.

---

### 5.1. Что такое MemorySaver?

LangGraph предоставляет встроенный механизм сохранения состояния — **checkpointers**. Каждый раз, когда граф проходит через узел, его состояние (включая все сообщения) сериализуется и сохраняется. При следующем вызове граф может восстановить состояние по идентификатору сессии.

Самый простой чекпоинтер — `MemorySaver` из `langgraph.checkpoint.memory`. Он хранит состояния в оперативной памяти (подходит для разработки и тестирования).

**Импорт:**

```python
from langgraph.checkpoint.memory import MemorySaver
```

---

### 5.2. Компиляция графа с чекпоинтером

Чтобы включить память, достаточно передать `checkpointer` при компиляции графа:

```python
# Создаём чекпоинтер
memory = MemorySaver()

# Компилируем граф с чекпоинтером
graph = builder.compile(checkpointer=memory)
```

Теперь граф будет автоматически сохранять состояние после каждого шага.

---

### 5.3. Вызов графа с идентификатором сессии (`thread_id`)

Чтобы восстановить состояние, нужно передать `thread_id` в конфигурации вызова. LangGraph использует этот идентификатор для поиска сохранённого состояния.

```python
# Первый вызов — создаётся новая сессия
config = {"configurable": {"thread_id": "user_123"}}

result1 = graph.invoke(
    {"messages": [("user", "Запомни, меня зовут Алекс")]},
    config=config
)

# Второй вызов — продолжение той же сессии
result2 = graph.invoke(
    {"messages": [("user", "Как меня зовут?")]},
    config=config
)

# Агент помнит имя, потому что состояние восстановлено
print(result2["messages"][-1].content)  # "Вас зовут Алекс."
```

**Важно:** `thread_id` может быть любым строковым идентификатором (имя пользователя, UUID, номер чата). Все вызовы с одинаковым `thread_id` разделяют общую историю.

---

### 5.4. Полный код агента с памятью (`agent_with_memory.py`)

Ниже представлен полностью рабочий код агента с памятью, который мы протестировали. Он включает два инструмента (`calculate` и `get_current_time`), чёткий системный промпт и защиту от пустых ответов.

```python
"""
agent_with_memory.py - Агент на LangGraph с памятью (MemorySaver)
Лекция 6.4, Тема 5
"""

import math
import re
from datetime import datetime
from typing import Annotated, List, TypedDict

from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# ============================================================================
# 1. ИНСТРУМЕНТЫ
# ============================================================================

@tool
def calculate(expression: str) -> str:
    """Выполняет математические вычисления."""
    try:
        safe_dict = {'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos, 'tan': math.tan,
                     'pi': math.pi, 'e': math.e, 'abs': abs, 'round': round}
        cleaned = re.sub(r'[^0-9+\-*/%().,sqrt sincostanlogpi eabsround]', '', expression.lower())
        result = eval(cleaned, {"__builtins__": {}}, safe_dict)
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {e}"

@tool
def get_current_time() -> str:
    """Возвращает текущие дату и время."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

tools = [calculate, get_current_time]

# ============================================================================
# 2. LLM С ИНСТРУМЕНТАМИ
# ============================================================================

SYSTEM_PROMPT = """
Ты — интеллектуальный агент с доступом к инструментам.

Инструменты:
- calculate: выполняет математические вычисления (пример: '2+2', 'sqrt(16)').
- get_current_time: возвращает текущие дату и время.

Правила работы:
1. Если вопрос требует вычислений — используй calculate.
2. Если вопрос о времени/дате — используй get_current_time.
3. Если вопрос НЕ требует вычислений или времени (например, запоминание имени, общий разговор) — НЕ вызывай инструменты, а просто ответь текстом.
4. После получения результата от инструмента, сформулируй чёткий и полный ответ пользователю.
5. Никогда не вызывай инструменты без необходимости.
"""

llm = ChatOllama(model="qwen2.5:3b", temperature=0.0, num_predict=512)
llm_with_tools = llm.bind_tools(tools)

# ============================================================================
# 3. СОСТОЯНИЕ
# ============================================================================

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

# ============================================================================
# 4. УЗЛЫ ГРАФА
# ============================================================================

def agent_node(state: AgentState) -> AgentState:
    messages = state["messages"]
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    response = llm_with_tools.invoke(messages)
    print(f"DEBUG agent_node: response = {response}")
    if hasattr(response, "tool_calls") and response.tool_calls:
        print(f"  -> tool_calls: {response.tool_calls}")
    else:
        print(f"  -> content: {response.content[:100]}...")
    return {"messages": [response]}

def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        print("DEBUG: should_continue -> tools")
        return "tools"
    print("DEBUG: should_continue -> final_answer")
    return "final_answer"

def final_answer_node(state: AgentState) -> AgentState:
    messages = state["messages"]
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    
    # Используем отдельную модель с большим num_predict для финального ответа
    llm_final = ChatOllama(model="qwen2.5:3b", temperature=0.0, num_predict=1024)
    response = llm_final.invoke(messages)
    
    # Если ответ пустой — используем последнее сообщение от агента
    if not response.content or len(response.content.strip()) < 2:
        for msg in reversed(messages):
            if isinstance(msg, type(response)) and msg.content:
                response.content = msg.content
                break
        else:
            response.content = "Извините, не удалось сформулировать ответ."
    
    print(f"DEBUG final_answer_node: {response.content[:100]}...")
    return {"messages": [response]}

tool_node = ToolNode(tools)

# ============================================================================
# 5. СБОРКА ГРАФА С ЧЕКПОИНТЕРОМ
# ============================================================================

builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_node("final_answer", final_answer_node)

builder.set_entry_point("agent")
builder.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", "final_answer": "final_answer"}
)
builder.add_edge("tools", "agent")
builder.add_edge("final_answer", END)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

# ============================================================================
# 6. ТЕСТИРОВАНИЕ
# ============================================================================

if __name__ == "__main__":
    session_id = "user_123"
    config = {"configurable": {"thread_id": session_id}, "recursion_limit": 10}

    questions = [
        "Запомни, меня зовут Алексей.",
        "Как меня зовут?",
        "Сколько будет 2+2?",
        "А теперь скажи моё имя ещё раз."
    ]

    for q in questions:
        print(f"\n{'='*60}")
        print(f"Вопрос: {q}")
        print('-'*60)

        result = graph.invoke(
            {"messages": [("user", q)]},
            config=config
        )

        last_msg = result["messages"][-1]
        print(f"Ответ: {last_msg.content}")
```

---

### 5.5. Результат тестирования

При запуске скрипта мы получаем следующий вывод:

```
============================================================
Вопрос: Запомни, меня зовут Алексей.
------------------------------------------------------------
DEBUG agent_node: response = content='Я запомнил, тебя зовут Алексей.' ...
DEBUG: should_continue -> final_answer
DEBUG final_answer_node: ...
Ответ: Я запомнил, тебя зовут Алексей.

============================================================
Вопрос: Как меня зовут?
------------------------------------------------------------
DEBUG agent_node: response = content='Вы сказали, что ваше имя Алексей.' ...
DEBUG: should_continue -> final_answer
DEBUG final_answer_node: ...
Ответ: Вы сказали, что ваше имя Алексей.

============================================================
Вопрос: Сколько будет 2+2?
------------------------------------------------------------
DEBUG agent_node: tool_calls = [{'name': 'calculate', 'args': {'expression': '2+2'}}]
DEBUG: should_continue -> tools
DEBUG agent_node: response = content='2+2 равно 4.' ...
DEBUG: should_continue -> final_answer
DEBUG final_answer_node: ...
Ответ: 2+2 равно 4.

============================================================
Вопрос: А теперь скажи моё имя ещё раз.
------------------------------------------------------------
DEBUG agent_node: response = content='Вы сказали, что ваше имя Алексей.' ...
DEBUG: should_continue -> final_answer
DEBUG final_answer_node: ...
Ответ: Вы сказали, что ваше имя Алексей.
```

**Что мы видим из вывода:**
- Агент запомнил имя после первого вопроса.
- На второй вопрос он вспомнил имя.
- На математический вопрос корректно вызвал инструмент `calculate`.
- На четвёртый вопрос снова правильно назвал имя.

Это значит, что **состояние сохранялось между вызовами** благодаря `MemorySaver` и `thread_id`.

---

### 5.6. Как это работает под капотом

Когда мы компилируем граф с `checkpointer`, каждый раз после выполнения **супер-шага** (то есть когда граф достигает узла, из которого нет выхода, например `END`) состояние сериализуется и сохраняется.

При следующем вызове с тем же `thread_id` граф:
1. Загружает сохранённое состояние.
2. Продолжает выполнение с того места, где остановился, или начинает новый диалог с уже накопленной историей.

Это отличается от ручного управления списком сообщений (как в Лекции 6.2) тем, что:
- **Сохраняется всё состояние**, а не только сообщения (можно хранить любые флаги, результаты промежуточных вычислений, контекст).
- **Автоматическое восстановление** — не нужно вручную подставлять историю в промпт.
- **Масштабируемость** — можно использовать другие чекпоинтеры (SQLite, PostgreSQL, Redis) для долгосрочного хранения.

---

### 5.7. Другие чекпоинтеры

`MemorySaver` удобен для разработки, но в продакшене лучше использовать:
- `SqliteSaver` — сохраняет состояние в SQLite (локально, персистентно).
- `PostgresSaver` — сохраняет в PostgreSQL (централизованно, масштабируемо).

Установка:
```bash
pip install langgraph-checkpoint-sqlite  # или postgres
```

Использование:
```python
from langgraph.checkpoint.sqlite import SqliteSaver

with SqliteSaver.from_conn_string("checkpoints.db") as memory:
    graph = builder.compile(checkpointer=memory)
```

---

## Краткий итог Тема 5

- **`MemorySaver`** — это встроенный чекпоинтер LangGraph, который сохраняет состояние графа в памяти.
- При компиляции графа передаём `checkpointer=MemorySaver()`.
- При вызове `graph.invoke()` указываем `config={"configurable": {"thread_id": "..."}}`.
- Все вызовы с одинаковым `thread_id` разделяют общую историю диалога.
- Это избавляет от ручного управления списком сообщений и делает код чище и надёжнее.
- Для продакшена можно использовать `SqliteSaver` или `PostgresSaver` для долгосрочного хранения.

---

**Теперь наш агент стал по-настоящему «диалоговым»: он запоминает контекст, может ссылаться на ранее сказанное и поддерживать длительные беседы. В следующей теме мы интегрируем агента в веб-интерфейс или Telegram-бота, чтобы он мог общаться с пользователем в реальном времени.** Оставайтесь с нами!


## Тема 6. Human-in-the-loop: контроль над агентом (скрипт `human_agent.py`)

Мы построили мощного агента, который умеет вызывать инструменты, анализировать результаты и запоминать контекст. Но есть одна важная деталь: **агент действует полностью автономно**. Он сам решает, какие инструменты вызывать и когда. В большинстве сценариев это нормально, но иногда нам нужно, чтобы человек мог **проверить и утвердить** действия агента перед их выполнением.

Представьте ситуацию:
- Агент хочет отправить email от вашего имени.
- Агент собирается выполнить дорогостоящий API-запрос.
- Агент планирует удалить файлы.
- Агент собирается опубликовать пост в социальных сетях.

В таких случаях мы хотим, чтобы агент **спросил разрешения** у человека перед выполнением действия. Это называется **Human-in-the-loop** — человек в цикле принятия решений.

---

### 6.1. Прерывания в LangGraph

LangGraph предоставляет встроенный механизм прерываний. При компиляции графа мы можем указать узлы, **перед которыми** или **после которых** выполнение должно останавливаться.

```python
# Компиляция с прерыванием перед узлом tools
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["tools"]  # Остановиться перед выполнением инструментов
)
```

Теперь каждый раз, когда граф достигает узла `tools`, выполнение приостанавливается, и управление возвращается пользователю.

**Что происходит:**

1. Агент (в узле `agent`) решает вызвать инструмент.
2. Граф доходит до узла `tools`.
3. Выполнение **останавливается**.
4. Мы можем посмотреть, что собирается делать агент.
5. Мы либо подтверждаем действие, либо отменяем, либо изменяем.
6. Затем мы запускаем продолжение, и граф выполняет инструмент.

---

### 6.2. Демонстрация Human-in-the-loop

Давайте добавим в нашего агента новый инструмент — отправку email. И сделаем так, чтобы перед его выполнением требовалось подтверждение пользователя.

**Полный код (`human_agent.py`):**

```python
"""
human_agent.py - Агент на LangGraph с Human-in-the-loop
Лекция 6.4, Тема 6
"""

import math
import re
from datetime import datetime
from typing import Annotated, List, TypedDict

from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# ============================================================================
# 1. ИНСТРУМЕНТЫ
# ============================================================================

@tool
def calculate(expression: str) -> str:
    """Выполняет математические вычисления."""
    try:
        safe_dict = {'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos, 'tan': math.tan,
                     'pi': math.pi, 'e': math.e, 'abs': abs, 'round': round}
        cleaned = re.sub(r'[^0-9+\-*/%().,sqrt sincostanlogpi eabsround]', '', expression.lower())
        result = eval(cleaned, {"__builtins__": {}}, safe_dict)
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {e}"

@tool
def get_current_time() -> str:
    """Возвращает текущие дату и время."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """
    Отправляет email указанному получателю.
    Используй этот инструмент, когда пользователь просит отправить письмо.
    Аргументы:
        to: адрес получателя
        subject: тема письма
        body: текст письма
    """
    # Имитация отправки
    return f"✅ Email отправлен на {to} с темой '{subject}'"

tools = [calculate, get_current_time, send_email]

# ============================================================================
# 2. LLM С ИНСТРУМЕНТАМИ
# ============================================================================

SYSTEM_PROMPT = """
Ты — интеллектуальный агент с доступом к инструментам.

Инструменты:
- calculate: математические вычисления.
- get_current_time: текущее время.
- send_email: отправка email (требует: to, subject, body).

Правила:
1. Если нужно посчитать — используй calculate.
2. Если нужно узнать время — используй get_current_time.
3. Если нужно отправить письмо — используй send_email.
4. После выполнения инструмента дай чёткий ответ пользователю.
"""

llm = ChatOllama(model="qwen2.5:3b", temperature=0.0, num_predict=512)
llm_with_tools = llm.bind_tools(tools)

# ============================================================================
# 3. СОСТОЯНИЕ
# ============================================================================

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

# ============================================================================
# 4. УЗЛЫ ГРАФА
# ============================================================================

def agent_node(state: AgentState) -> AgentState:
    messages = state["messages"]
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "final_answer"

def final_answer_node(state: AgentState) -> AgentState:
    messages = state["messages"]
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    
    # Для финального ответа используем увеличенный лимит токенов
    llm_final = ChatOllama(model="qwen2.5:3b", temperature=0.0, num_predict=1024)
    response = llm_final.invoke(messages)
    
    # Защита от пустого ответа
    if not response.content or len(response.content.strip()) < 2:
        # Ищем последнее содержательное сообщение от агента
        for msg in reversed(messages):
            if isinstance(msg, type(response)) and msg.content:
                response.content = msg.content
                break
        else:
            response.content = "Извините, не удалось сформулировать ответ."
    
    return {"messages": [response]}

tool_node = ToolNode(tools)

# ============================================================================
# 5. СБОРКА ГРАФА С ПРЕРЫВАНИЕМ
# ============================================================================

builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_node("final_answer", final_answer_node)

builder.set_entry_point("agent")
builder.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", "final_answer": "final_answer"}
)
builder.add_edge("tools", "agent")
builder.add_edge("final_answer", END)

# КОМПИЛЯЦИЯ С MEMORYSAVER И ПРЕРЫВАНИЕМ ПЕРЕД tools
memory = MemorySaver()
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["tools"]  # <--- КЛЮЧЕВОЙ МОМЕНТ
)

# ============================================================================
# 6. ТЕСТИРОВАНИЕ HUMAN-IN-THE-LOOP
# ============================================================================

def print_state(state):
    """Вспомогательная функция для вывода состояния."""
    messages = state.get("messages", [])
    print("\n📋 Состояние:")
    for msg in messages:
        if msg.type == "human":
            print(f"  👤 Пользователь: {msg.content}")
        elif msg.type == "ai":
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  🤖 Агент хочет вызвать: {tc['name']}({tc['args']})")
            else:
                preview = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
                print(f"  🤖 Агент: {preview}")
        elif msg.type == "tool":
            preview = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
            print(f"  🔧 Результат: {preview}")

if __name__ == "__main__":
    session_id = "user_123"
    config = {"configurable": {"thread_id": session_id}, "recursion_limit": 10}

    question = "Отправь письмо на alex@example.com с темой 'Привет' и текстом 'Как дела?'"

    print("=" * 60)
    print(f"Вопрос: {question}")
    print("=" * 60)

    # ШАГ 1: Запускаем граф — он остановится перед tools
    print("\n🚀 Запуск агента...")
    result = graph.invoke(
        {"messages": [("user", question)]},
        config=config
    )

    # ШАГ 2: Показываем, что агент собирается сделать
    print("\n⏸️ АГЕНТ ОСТАНОВЛЕН!")
    print_state(result)

    # ШАГ 3: Проверяем, что именно агент хочет вызвать
    last_message = result["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        print("\n📌 Агент планирует вызвать инструмент(ы):")
        for tc in last_message.tool_calls:
            print(f"   - {tc['name']} с параметрами: {tc['args']}")

        # ШАГ 4: Спрашиваем разрешения у пользователя
        user_input = input("\n✅ Разрешить выполнение? (y/n): ")

        if user_input.lower() == "y":
            print("\n▶️ Продолжаем выполнение...")
            # Продолжаем выполнение с того же состояния
            result = graph.invoke(None, config=config)
            print("\n✅ ГРАФ ЗАВЕРШЁН!")
            print_state(result)

            last_msg = result["messages"][-1]
            print(f"\n📧 Финальный ответ: {last_msg.content}")
        else:
            print("\n❌ Выполнение отменено пользователем.")

            # Добавляем сообщение об отмене
            cancel_state = {
                "messages": result["messages"] + [
                    HumanMessage(content="Пользователь отменил выполнение.")
                ]
            }
            result = graph.invoke(cancel_state, config=config)
            last_msg = result["messages"][-1]
            print(f"\n📧 Ответ: {last_msg.content}")
```

---

### 6.3. Результат работы

При запуске скрипта мы получаем:

```
============================================================
Вопрос: Отправь письмо на alex@example.com с темой 'Привет' и текстом 'Как дела?'
============================================================

🚀 Запуск агента...

⏸️ АГЕНТ ОСТАНОВЛЕН!

📋 Состояние:
  👤 Пользователь: Отправь письмо на alex@example.com с темой 'Привет' и текстом 'Как дела?'
  🤖 Агент хочет вызвать: send_email({'to': 'alex@example.com', 'subject': 'Привет', 'body': 'Как дела?'})

📌 Агент планирует вызвать инструмент(ы):
   - send_email с параметрами: {'to': 'alex@example.com', 'subject': 'Привет', 'body': 'Как дела?'}

✅ Разрешить выполнение? (y/n): y

▶️ Продолжаем выполнение...

✅ ГРАФ ЗАВЕРШЁН!

📋 Состояние:
  👤 Пользователь: Отправь письмо на alex@example.com с темой 'Привет' и текстом 'Как дела?'
  🤖 Агент хочет вызвать: send_email({'to': 'alex@example.com', ...})
  🔧 Результат: ✅ Email отправлен на alex@example.com с темой 'Привет'...
  🤖 Агент: Письмо успешно отправлено на адрес alex@example.com.

📧 Финальный ответ: Письмо успешно отправлено на адрес alex@example.com.
```

**Что произошло:**

1. Агент получил задачу отправить email.
2. Он решил вызвать инструмент `send_email`.
3. Граф остановился **перед** выполнением инструмента.
4. Мы увидели, что агент собирается сделать.
5. Мы подтвердили выполнение.
6. Граф продолжил работу, отправил email и дал ответ.

Если бы мы ввели `n`, выполнение было бы отменено, и агент не отправил бы письмо.

---

### 6.4. Практическая польза Human-in-the-loop

| Сценарий | Без прерывания | С прерыванием |
|----------|---------------|---------------|
| **Отправка email** | Агент отправит сразу | Вы можете проверить адресата и текст |
| **Дорогой API-запрос** | Агент выполнит и спишет средства | Вы можете подтвердить траты |
| **Удаление данных** | Агент может случайно удалить важное | Вы можете предотвратить удаление |
| **Публикация в соцсетях** | Пост уйдёт без вашего ведома | Вы можете отредактировать текст |
| **Отладка** | Трудно понять, что делает агент | Вы видите каждый шаг |

---

## Краткий итог Тема 6

- **Human-in-the-loop** реализуется через `interrupt_before` или `interrupt_after` при компиляции графа.
- При остановке графа мы можем проверить, какие инструменты собирается вызвать агент, и принять решение.
- Это повышает **безопасность, контроль и доверие** к агенту.
- В коде мы добавили защиту от пустого ответа в `final_answer_node`, чтобы финальный ответ всегда был осмысленным.

---

**Теперь наш агент стал не только умным, но и безопасным: он спрашивает разрешения перед опасными действиями.**

## Домашнее задание к лекции 6.4

В этой лекции мы перешли от линейных цепочек к графам состояний и построили **настоящего агента** на LangGraph. Наш агент умеет циклически принимать решения: анализировать вопрос, вызывать инструменты (калькулятор, время, веб-поиск, поиск в документах), анализировать результаты и повторять процесс, пока не будет готов ответ. Мы добавили память (MemorySaver) для сохранения истории диалога между запросами и механизм Human‑in‑the‑loop для безопасного контроля над действиями агента.

Ваша задача — закрепить эти концепции на практике: провести сравнительные эксперименты, исследовать поведение агента, модифицировать его и сделать обоснованные выводы.

> **Важно:** Все задания выполняются с использованием кода из лекции (`agent_graph.py`, `agent_with_memory.py`, `human_agent.py`) как основы. Вы можете модифицировать эти скрипты, добавлять новые инструменты, менять системные промпты и параметры. Для экспериментов используйте локальную модель Ollama (например, `qwen2.5:3b`). Если у вас есть собственные документы, подключите реальный ретривер к инструменту `search_docs` (как описано в теме 4.10) для более содержательных тестов.

---

### Обязательная часть (5 баллов)

#### 1. Сравнение агента на LangGraph и обычного RAG-пайплайна (2 балла)

В предыдущих лекциях мы строили линейные RAG-цепочки (например, в Лекции 6.3). Агент на LangGraph умеет вызывать несколько инструментов, переключаться между ними и повторять шаги. Ваша задача — количественно сравнить эти два подхода на наборе вопросов, требующих нелинейного решения.

**Что сделать:**

1. Возьмите **линейный RAG-пайплайн** из Лекции 6.3 (или 6.2) — тот, который просто ищет в документах и генерирует ответ. Убедитесь, что он использует тот же ретривер, что и ваш агент (если вы подключали реальный поиск).

2. Подготовьте **набор из 12 вопросов** трёх типов:
   - 4 вопроса, требующих **вычислений** (например, «Сколько будет 15% от 200?»).
   - 4 вопроса, требующих **поиска в документах** (например, «Что такое ProjectFlow?»).
   - 4 вопроса, требующих **комбинации действий** (например, «Найди в документах выручку за прошлый год и вычисли её рост на 10%»).

3. Для каждого вопроса запустите:
   - **Линейный пайплайн** (только RAG, без вычислений и других инструментов).
   - **Агента на LangGraph** (с инструментами `calculate`, `search_docs`, `web_search` и т.д.).

   Зафиксируйте для каждого запуска:
   - Финальный ответ.
   - Количество шагов (для агента — число итераций цикла; для пайплайна — всегда 1).
   - Время выполнения.
   - Наличие ошибок (например, агент не смог найти ответ, или пайплайн выдал галлюцинацию).

4. **В отчёте**:
   - Приведите таблицу сравнения по всем вопросам (столбцы: вопрос, пайплайн-ответ, агент-ответ, шаги, время).
   - Для каждой группы вопросов (вычисления, поиск, комбинация) рассчитайте **среднее время** и **долю успешных ответов**.
   - Проанализируйте, в каких случаях агент дал лучшее качество, а в каких — пайплайн был достаточен.
   - Сделайте вывод: для каких задач стоит использовать агента, а где достаточно простого RAG.

---

#### 2. Анализ цикла ReAct: количество итераций и качество решений (3 балла)

Агент работает по принципу ReAct (Reasoning + Acting): он чередует «размышление» (узел `agent`) и «действие» (узел `tools`). Ваша задача — исследовать, как агент принимает решения, сколько итераций ему требуется, и как это зависит от формулировки вопроса и системного промпта.

**Что сделать:**

1. Модифицируйте код `agent_graph.py` (или `agent_with_memory.py`) так, чтобы **логировать** каждый шаг цикла:
   - Номер итерации.
   - Решение агента (вызов инструментов или финальный ответ).
   - Какие инструменты были вызваны и с какими аргументами.
   - Содержание финального ответа.

   Вы можете использовать встроенные `print` или кастомное логирование.

2. Подготовьте **набор из 10 вопросов**, которые требуют **одного** и **нескольких** вызовов инструментов:
   - 5 вопросов, на которые агент должен ответить без инструментов (например, «Как тебя зовут?» или «Привет»).
   - 5 вопросов, требующих инструментов (например, вычисления, поиск времени, комбинированные).

3. Запустите агента на этих вопросах и для каждого запишите:
   - Количество итераций.
   - Последовательность вызванных инструментов.
   - Был ли случай, когда агент вызвал инструмент, но потом всё равно ответил неверно или неполно.

4. Теперь **измените системный промпт** агента (`SYSTEM_PROMPT`) двумя способами:
   - **Сделайте промпт более строгим** — добавьте правило: «Никогда не вызывай инструменты, если вопрос можно решить без них».
   - **Сделайте промпт более лояльным** — добавьте: «При малейшем сомнении используй инструменты для проверки».

   Для каждого варианта промпта снова запустите те же 10 вопросов и запишите число итераций и качество ответов.

5. **В отчёте**:
   - Приведите таблицу с количеством итераций для каждого вопроса для трёх конфигураций (исходный, строгий, лояльный).
   - Проанализируйте, как промпт влияет на число вызовов инструментов и на качество ответов.
   - Приведите примеры, когда агент вызвал инструмент без необходимости (или, наоборот, не вызвал, хотя надо было).
   - Сделайте вывод: какую стратегию промпта вы бы рекомендовали для ваших задач и почему.

---

### Дополнительная часть (эксперименты — выполните минимум 3 из 6, каждый до 2 баллов)

#### 3. Добавление нового инструмента (2 балла)

В лекции мы использовали четыре инструмента: `calculate`, `get_current_time`, `web_search`, `search_docs`. Добавьте **новый инструмент**, например, конвертер валют, генератор случайных чисел, или инструмент для работы с файлами.

**Что сделать:**

1. Реализуйте новый инструмент с помощью декоратора `@tool`, опишите его назначение в docstring.
2. Добавьте его в список `tools` и привяжите к модели через `bind_tools`.
3. Протестируйте агента на **5 вопросах**, где новый инструмент должен быть вызван (например, «Сконвертируй 100 долларов в евро»).
4. **В отчёте**:
   - Приведите код нового инструмента и его описание.
   - Покажите примеры вопросов и последовательность вызовов, подтверждающую, что агент использует новый инструмент.
   - Оцените, насколько легко агенту «понять», когда использовать новый инструмент (т.е. правильно ли он его выбирает).

---

#### 4. Эксперименты с параметрами модели (температура, num_predict) (2 балла)

Агент использует модель `qwen2.5:3b` с параметрами по умолчанию (`temperature=0.0`, `num_predict=512`). Измените эти параметры и посмотрите, как это влияет на выбор инструментов и качество ответов.

**Что сделать:**

1. Создайте три конфигурации модели:
   - **Строгая**: `temperature=0.0`, `num_predict=256`.
   - **Креативная**: `temperature=0.7`, `num_predict=512`.
   - **Длинная**: `temperature=0.0`, `num_predict=1024`.

2. Для каждой конфигурации запустите агента на **одном и том же наборе из 8 вопросов** (включая вычисления, поиск, общие вопросы). Зафиксируйте:
   - Частоту вызовов инструментов (сколько раз агент использовал каждый инструмент).
   - Полноту и точность финальных ответов (оцените вручную).
   - Время выполнения.

3. **В отчёте**:
   - Приведите таблицу сравнения по всем вопросам для трёх конфигураций.
   - Проанализируйте, как температура влияет на склонность агента вызывать инструменты (например, при высокой температуре агент может «фантазировать» и реже обращаться к инструментам).
   - Сделайте вывод, какие параметры оптимальны для вашего сценария.

---

#### 5. Расширение Human‑in‑the‑loop (2 балла)

В лекции мы реализовали прерывание перед узлом `tools`. Расширьте этот механизм: добавьте возможность не только подтвердить/отклонить вызов, но и **изменить аргументы** инструмента перед выполнением.

**Что сделать:**

1. Модифицируйте код `human_agent.py` так, чтобы при остановке перед `tools` пользователь мог:
   - Ввести `y` — подтвердить с текущими аргументами.
   - Ввести `e` — редактировать аргументы (ввести новое значение для каждого параметра).
   - Ввести `n` — отменить.

2. Протестируйте на **трёх сценариях**, где изменение аргументов имеет смысл (например, изменить адресата письма или выражение для вычисления).

3. **В отчёте**:
   - Опишите, как вы реализовали редактирование.
   - Приведите примеры диалогов с пользователем, показывающие, как он меняет аргументы.
   - Оцените, насколько такая возможность повышает полезность системы.

---

#### 6. Сравнение MemorySaver и отсутствия памяти (2 балла)

В лекции мы добавили память через `MemorySaver`. Проверьте, как агент ведёт себя без памяти и с памятью в диалоге из нескольких вопросов.

**Что сделать:**

1. Создайте две версии агента: с `MemorySaver` и без него (просто `compile()` без checkpointer).

2. Подготовьте **диалог из 5 вопросов**:
   - Вопрос 1: «Запомни, меня зовут Иван».
   - Вопрос 2: «Как меня зовут?» (агент должен вспомнить имя).
   - Вопрос 3: «Сколько будет 2+2?» (проверка, что память не мешает вычислениям).
   - Вопрос 4: «А теперь скажи моё имя ещё раз» (проверка долгосрочной памяти).
   - Вопрос 5: «Сколько будет 10% от 200?» (ещё одна проверка).

3. Запустите оба варианта на этом диалоге (с одним `thread_id` для версии с памятью, и без `thread_id` для версии без памяти). Запишите ответы на все вопросы.

4. **В отчёте**:
   - Приведите полные диалоги для обеих версий.
   - Проанализируйте, на каких вопросах разница особенно заметна.
   - Обсудите, есть ли случаи, когда память вредит (например, агент «зацикливается» на предыдущей теме).

---

#### 7. Добавление ограничения на число итераций и обработка ошибок (2 балла)

Агент может зациклиться, если не может принять решение. В лекции мы использовали `recursion_limit` в конфиге. Ваша задача — исследовать, что происходит, когда агент превышает лимит, и как это можно обработать.

**Что сделать:**

1. Установите `recursion_limit=3` (очень маленькое значение) и задайте вопрос, требующий нескольких шагов (например, «Найди в документах выручку и вычисли её рост на 20%», что потребует поиска и вычислений).

2. Запустите агента и зафиксируйте ошибку `GraphRecursionError`.

3. Обработайте эту ошибку в коде (try/except) и верните понятное сообщение пользователю, например: «Агент не смог завершить задачу за отведённое время. Попробуйте упростить вопрос».

4. Затем увеличьте лимит до 10 и снова запустите тот же вопрос, чтобы убедиться, что агент справляется.

5. **В отчёте**:
   - Опишите, как вы обработали ошибку.
   - Приведите пример вывода с ошибкой и с корректным завершением.
   - Сделайте вывод, какое значение `recursion_limit` является оптимальным для ваших типовых вопросов.

---

### Формат сдачи

1. **Код** всех модификаций сохраните в отдельной папке (или используйте Git). Убедитесь, что код хорошо задокументирован и легко запускается.

2. **Отчёт** в формате `.md` или `.pdf`, содержащий:
   - Для **каждого выполненного задания**:
     - Краткое описание, что сделано.
     - Ключевые результаты (таблицы, графики, примеры ответов, фрагменты кода).
     - Ваши выводы и обоснования.
   - Для обязательной части — чёткие сравнения и численные оценки.
   - Для дополнительной части — достаточные детали, чтобы можно было воспроизвести эксперимент.

3. **Лог-файлы** (по желанию) — приложите фрагменты логов для подтверждения экспериментов (особенно для задания 2).

---

### Критерии оценки

| Раздел | Максимум баллов |
|--------|----------------|
| **Обязательная часть** | 5 |
| Задание 1 (сравнение с RAG-пайплайном) | 2 |
| Задание 2 (анализ цикла ReAct) | 3 |
| **Дополнительная часть** (каждый пункт) | до 2 (максимум +10) |
| За каждый выполненный пункт: 1 балл за реализацию, 1 балл за анализ/выводы | |
| **Итого** | 15 |

**Штрафы:**
- Отсутствие отчёта или невнятные выводы — минус 2 балла.
- Код без комментариев и с плохим форматированием — минус 1 балл.
- Использование готовых ответов без собственного анализа — минус 2 балла.

---

### Рекомендации

- Начинайте с обязательной части — она даёт базовое понимание работы агента.
- Для экспериментов используйте **один и тот же набор вопросов**, чтобы результаты были сопоставимы.
- Вносите изменения в код аккуратно, делайте резервные копии.
- Активно используйте логирование — оно поможет понять, что происходит внутри графа.
- В выводах опирайтесь не только на численные данные, но и на качественный анализ поведения агента.
- Если у вас нет реальных документов, используйте заглушку `search_docs` из лекции или создайте простые тестовые документы.
